# 05 — Power BI Preparation

Objectif : préparer des tables propres et exploitables dans Power BI à partir de la table `sales_featured.csv`.

Nous allons créer un modèle simple en étoile :

- `fact_sales.csv` : table centrale des ventes
- `dim_customers.csv` : informations clients
- `dim_products.csv` : informations produits
- `dim_sellers.csv` : informations vendeurs
- `dim_dates.csv` : calendrier
- `kpi_summary.csv` : indicateurs clés pour contrôle rapide

Ces fichiers seront exportés dans `data/powerbi/`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Chargement des données

In [2]:
DATA_PROCESSED = Path("data/processed")
POWERBI_PATH = Path("data/powerbi")
POWERBI_PATH.mkdir(parents=True, exist_ok=True)

sales = pd.read_csv(
    DATA_PROCESSED / "sales_featured.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "shipping_limit_date",
        "review_creation_date",
        "review_answer_timestamp"
    ]
)

sales.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data\\processed\\sales_featured.csv'

In [ ]:
print(sales.shape)
sales.columns.tolist()

## 2. Nettoyage final pour Power BI

On évite les valeurs nulles gênantes dans les dimensions et on prépare quelques colonnes plus lisibles.

In [ ]:
sales["product_category_name"] = sales["product_category_name"].fillna("unknown")
sales["review_label"] = sales["review_label"].fillna("Unknown")
sales["customer_state"] = sales["customer_state"].fillna("Unknown")
sales["seller_state"] = sales["seller_state"].fillna("Unknown")

# Pour éviter les divisions ou affichages étranges dans Power BI
sales["freight_ratio"] = sales["freight_ratio"].replace([np.inf, -np.inf], np.nan).fillna(0)

sales.head()

## 3. Création de la table `fact_sales`

La table de faits contient les mesures principales : chiffre d'affaires, frais de livraison, délais, notes, statut de livraison, etc.

In [ ]:
fact_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id",
    "product_id",
    "seller_id",
    "order_purchase_timestamp",
    "order_status",
    "price",
    "freight_value",
    "total_order_value",
    "payment_value",
    "payment_type",
    "payment_installments",
    "review_score",
    "review_label",
    "delivery_time_days",
    "delivery_delay_days",
    "approval_time_hours",
    "shipping_time_days",
    "freight_ratio",
    "items_per_order",
    "is_delayed",
    "purchase_year",
    "purchase_month",
    "purchase_quarter",
    "purchase_day",
    "purchase_weekday",
    "purchase_hour"
]

fact_sales = sales[[col for col in fact_columns if col in sales.columns]].copy()

fact_sales.head()

In [ ]:
fact_sales.shape

## 4. Création de `dim_customers`

In [ ]:
dim_customers = (
    sales[[
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_customers.head()

In [ ]:
dim_customers.shape

## 5. Création de `dim_products`

In [ ]:
product_cols = [
    "product_id",
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

dim_products = (
    sales[[col for col in product_cols if col in sales.columns]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_products.head()

In [ ]:
dim_products.shape

## 6. Création de `dim_sellers`

In [ ]:
seller_cols = [
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state"
]

dim_sellers = (
    sales[[col for col in seller_cols if col in sales.columns]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_sellers.head()

In [ ]:
dim_sellers.shape

## 7. Création de `dim_dates`

Cette table calendrier sera utile dans Power BI pour créer des slicers, des tendances mensuelles et des analyses temporelles.

In [ ]:
min_date = sales["order_purchase_timestamp"].min().date()
max_date = sales["order_purchase_timestamp"].max().date()

date_range = pd.date_range(start=min_date, end=max_date, freq="D")

dim_dates = pd.DataFrame({"date": date_range})
dim_dates["year"] = dim_dates["date"].dt.year
dim_dates["quarter"] = dim_dates["date"].dt.quarter
dim_dates["month"] = dim_dates["date"].dt.month
dim_dates["month_name"] = dim_dates["date"].dt.month_name()
dim_dates["day"] = dim_dates["date"].dt.day
dim_dates["weekday"] = dim_dates["date"].dt.day_name()
dim_dates["week"] = dim_dates["date"].dt.isocalendar().week.astype(int)
dim_dates["year_month"] = dim_dates["date"].dt.to_period("M").astype(str)

dim_dates.head()

In [ ]:
dim_dates.shape

## 8. Création de `kpi_summary`

Cette table sert à vérifier rapidement les chiffres dans Power BI.

In [ ]:
kpi_summary = pd.DataFrame({
    "kpi": [
        "Total Revenue",
        "Total Orders",
        "Total Customers",
        "Average Order Value",
        "Delayed Orders Rate",
        "Average Review Score",
        "Average Delivery Time"
    ],
    "value": [
        sales["total_order_value"].sum(),
        sales["order_id"].nunique(),
        sales["customer_unique_id"].nunique(),
        sales["total_order_value"].sum() / sales["order_id"].nunique(),
        sales["is_delayed"].mean() * 100,
        sales["review_score"].mean(),
        sales["delivery_time_days"].mean()
    ]
})

kpi_summary

## 9. Export des fichiers pour Power BI

In [ ]:
fact_sales.to_csv(POWERBI_PATH / "fact_sales.csv", index=False)
dim_customers.to_csv(POWERBI_PATH / "dim_customers.csv", index=False)
dim_products.to_csv(POWERBI_PATH / "dim_products.csv", index=False)
dim_sellers.to_csv(POWERBI_PATH / "dim_sellers.csv", index=False)
dim_dates.to_csv(POWERBI_PATH / "dim_dates.csv", index=False)
kpi_summary.to_csv(POWERBI_PATH / "kpi_summary.csv", index=False)

print("Fichiers exportés dans :", POWERBI_PATH)
print(list(POWERBI_PATH.glob("*.csv")))

## 10. Contrôle final

In [ ]:
exports = {
    "fact_sales": fact_sales,
    "dim_customers": dim_customers,
    "dim_products": dim_products,
    "dim_sellers": dim_sellers,
    "dim_dates": dim_dates,
    "kpi_summary": kpi_summary
}

for name, df in exports.items():
    print(f"{name}: {df.shape}")

# Prochaine étape

Importer ces fichiers CSV dans Power BI puis construire un modèle relationnel :

- `fact_sales[customer_id]` → `dim_customers[customer_id]`
- `fact_sales[product_id]` → `dim_products[product_id]`
- `fact_sales[seller_id]` → `dim_sellers[seller_id]`
- `fact_sales[order_purchase_timestamp]` → `dim_dates[date]` après création d'une colonne Date dans Power BI ou dans Python.

Pages recommandées du dashboard :

1. Executive Overview
2. Sales Analysis
3. Product Performance
4. Customer & Geography
5. Logistics & Delivery
6. Seller Performance

In [6]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from src.data_loader import load_raw_datasets, create_sales_table

DATA_RAW = Path("data/raw")

datasets = load_raw_datasets(DATA_RAW)
sales = create_sales_table(datasets)

sales.shape

NameError: name 's' is not defined